## KRP - PC application
- PC Application for the USB project on the STM32H747I-DISCO board
- Github: https://github.com/pavlio12/KRP_Project


In [ ]:
from connection_manager import ConnectionManager
from sender import sender_loop
from gui import create_gui

from queue import Queue
import threading
from collections import deque
import threading, time
from IPython.display import Javascript, display

message_queue = Queue()  # Thread-safe queue for messages to send

### Configuration

In [ ]:
# === Configuration ===
PORT = "COM18"      # Windows example
BAUDRATE = 115200   # default CDC speed

In [ ]:
# ----- LOG STATE (one buffer, one lock) -----
_LOG_MAX = 500  # keep it bounded so the notebook doesn’t bloat
_log_lines = deque(maxlen=_LOG_MAX)
_log_lock = threading.Lock()

### Main execution

In [ ]:
def main():
    # === GUI Callbacks ===
    def on_status_change(msg):
        """Update the status label in the GUI."""
        status_label.value = msg

    # def on_receive(msg):
    #     """Display received messages in the output area."""
    #     output_area.append_stdout(f"[Rx] {msg}\n")

    # def on_tx(msg):
    #     """Display transmitted messages in the output area."""
    #     output_area.append_stdout(f"[Tx] {msg}\n")

    def on_receive(msg):
        ts = time.strftime("%H:%M:%S")
        output_area.append_stdout(f"[{ts}] RX {msg}\n")
        _scroll_to_bottom()
    def on_tx(msg):
        ts = time.strftime("%H:%M:%S")
        output_area.append_stdout(f"[{ts}] TX {msg}\n")
        _scroll_to_bottom()

    def _scroll_to_bottom():
        display(Javascript("""
        (function(){
            var out = document.querySelector('.auto_scroll_output .output_subarea');
            if (out) { out.scrollTop = out.scrollHeight; }
        })();
        """))
        
    # === Connection Management ===
    def start_connection():
        global conn_manager
        conn_manager = ConnectionManager(PORT, BAUDRATE, on_status_change, on_receive)
        conn_manager.start()
        threading.Thread(target=sender_loop, args=(conn_manager, message_queue, on_tx), daemon=True).start()

    def stop_connection():
        global conn_manager
        if conn_manager:
            conn_manager.stop()
            conn_manager = None  # Clear the reference to avoid reuse
            status_label.value = "Disconnected"

    # === GUI Initialization ===
    gui, status_label, output_area = create_gui(start_connection, stop_connection, message_queue.put)
    display(gui)

main()



<IPython.core.display.Javascript object>